In [2]:
#! /usr/bin/env python
#! -*- coding=utf-8 -*-
# Project:  Lihang
# Filename: dt
# Date: 10/4/18
# Author: 😏 <smirk dot cao at gmail dot com>
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

In [3]:
tx = np.array([1,2,3,2,2,1,2,2,3,3,3,4,4,10,12,11,1,1,2,3,3,5,5,3,3,3,6,6,6,6,6,6,6,6,6,6,6])
li = [(i, len(list(filter(lambda tmp: tmp == i, tx)))) for i in set(tx)]
max_val = max(li, key=lambda tmp: tmp[1])
print(max_val)

(np.int64(6), 11)


In [ ]:
class Tree(object):

    def __init__(self,
                 eps,
                 feas,
                 name=None,
                 criterion="entropy"):

        self.tree_ = dict()
        # 特征数量
        self.feas = feas
        self.eps = eps
        self.criterion = criterion
        if not name:
            self.name = "Decision Tree"
        else:
            self.name = name

    def fit(self, x, y):
        self.tree_ = self._build_tree(x, y, self.eps)
        return self.tree_

    def predict(self, x, x_tree=None):
        if len(x.shape) == 2:
            rst = []
            for x_ in x:
                rst.append(self.predict(x_))
            return rst

        if not x_tree:
            x_tree = self.tree_
        tree_key = list(x_tree.keys())[0]
        x_fea = tree_key.split("__")[0]
        x_idx = clf.feas.index(x_fea)
        x_tree = x_tree[tree_key]
        for key in x_tree.keys():
            if key.split("__")[0] == x[x_idx]:
                tree_key = key
                x_tree = x_tree[tree_key]
        if type(x_tree) == dict:
            return self.predict(x, x_tree)
        else:
            return x_tree

    def _build_tree(self, x, y, eps):
        # 特征数
        feas = np.arange(x.shape[1])
        labels = y

        # step1: same label
        if len(set(labels)) == 1:
            return labels[0]

        # 最多数量类别
        max_label = max([(i, len(list(filter(lambda tmp: tmp == i, labels)))) for i in set(labels)]
                        , key=lambda tmp: tmp[1])[0]

        # step2: empty features
        if len(feas) == 0:
            return max_label

        # step3: feature selection
        max_fea = 0
        max_criterion = 0
        D = labels
        for fea in feas:
            # 特征选择
            A = x[:, fea]
            if self.criterion == "entropy":
                gda = gain(A, D)
            elif self.criterion == "gr":
                gda = gain_ratio(A, D)
            elif self.criterion == "gini":
                pass
            # uncomment this line for ex 5.3 gda result check
            # print(gda)
            if max_criterion < gda: # 当前特征的信息增益选择
                max_criterion, max_fea = gda, fea

        # step4: eps
        if max_criterion < eps: # 信息增益（比）阈值
            return max_label
        
        T = dict()
        sub_T = dict()
        for x_A in set(x[:, max_fea]):
            # 类别取值
            sub_D = D[x[:, max_fea] == x_A]
            # 特征取值
            sub_x = x[x[:, max_fea] == x_A, :]
            # 删除选择的特征列
            sub_x = np.delete(sub_x, max_fea, 1)
            # step6:
            sub_T[str(x_A) + "__" + str(sub_D.shape[0])] = self._build_tree(sub_x, sub_D, eps)
        # step5: T
        # self.tree_[max_fea] = sub_tree
        T[str(self.feas[max_fea]) + "__" + str(D.shape[0])] = sub_T
        return T

    def describe_tree(self, tree=None):
        rst = []
        if not tree:
            tree = self.tree_
        for fea_idx in tree.keys():
            tmp = dict()
            tmp["name"] = fea_idx.split("__")[0]
            tmp["value"] = fea_idx.split("__")[1]
            if type(tree[fea_idx]) == dict:
                tmp["children"] = self.describe_tree(tree[fea_idx])
            else:
                tmp["children"] = [{"name": tree[fea_idx], "value": 10}]
            rst.append(tmp)
        return rst

    def plot_tree(self, depth=3):
        from pyecharts import TreeMap

        data = self.describe_tree(self.tree_)
        tree_map = TreeMap(self.name, "", width=800, height=500)
        tree_map.use_theme("dark")
        tree_map.add(self.name, data, is_label_show=True, label_pos='inside', treemap_left_depth=depth)
        return tree_map

    def _choose_best_fea(self, x_, y_):
        rst = []
        for i in range(x_.shape[1]):
            for s_value in set(x_[:, i]):
                rst.append((i, s_value, gini(x_[:, i], y_=y_, s_=s_value)))
        rst.sort(key=lambda x: x[2])
        return rst[0] if rst != [] else rst

    def _build_cart(self, x_, y_):
        """

        :param x_:
        :param y_:
        :return: cart_tree
        """
        cart = dict()
        fea = self._choose_best_fea(x_, y_)
        if len(fea) > 0:
            key = fea[:2]
            idx_l = x_[:, fea[0]] == fea[1]
            idx_r = x_[:, fea[0]] != fea[1]
            if len(set(y_)) == 1:
                cart = y_[0]
            elif fea[2] == 0:
                cart[key] = {"left": y_[idx_l][0], "right": y_[idx_r][0]}
            else:

                if x_.shape[1] > 1:
                    x_l = x_[idx_l, :]
                    x_r = x_[idx_r, :]
                else:
                    x_l = x_[idx_l]
                    x_r = x_[idx_r]
                y_l = y_[idx_l]
                y_r = y_[idx_r]
                cart[key] = {"left": self._build_cart(x_l, y_l),
                             "right": self._build_cart(x_r, y_r)}
        else:
            pass
        return cart

    def create_cart(self, x_, y_):
        return self._build_cart(x_, y_)

    def pruning(self):
        # TODO: Pruning
        pass


def gini(x_, y_=None, s_=None):
    """

    :param x_: Feature A
    :param y_: Class D
    :param s_: split threshold
    :return: gini(y,x) or gini(y)
    """
    if y_ is None:
        x_values = list(set(x_))
        p = 0
        for x_value in x_values:
            p += (x_[x_ == x_value].shape[0] / x_.shape[0]) ** 2
        return 1 - p
    else:
        D1 = y_[x_ == s_]
        D2 = y_[x_ != s_]
        rst = D1.shape[0]/y_.shape[0]*gini(D1)+D2.shape[0]/y_.shape[0]*gini(D2)
        return rst


def cal_ent(x): # 信息熵
    """
    calculate shannon ent of x
    :param x:
    :return ent: H(D)=-\sum_{k=1}^K\frac{|C_k|}{|D|}\log_2\frac{|C_k|}{|D|}
    """
    x_values = list(set(x))
    ent = 0
    for x_value in x_values:
        p = x[x == x_value].shape[0]/x.shape[0]
        ent -= p*np.log2(p)
    return ent


def cal_condition_ent(x, y): # 条件熵
    """
    calculate condition ent(y|x)
    :param x: feature
    :param y: class
    :return: ent(y|x)
    """
    ent = 0
    x_values = set(x)
    for x_value in x_values:
        sub_y = y[x == x_value]
        tmp_ent = cal_ent(sub_y)
        p = sub_y.shape[0]/y.shape[0]
        ent += p*tmp_ent
    return ent


def gain(x, y): # 信息增益
    """
    calculate information g(y, x)
    :param x: feature
    :param y: class
    :return: gain
    """
    return cal_ent(y) - cal_condition_ent(x, y)


def gain_ratio(x, y): # 信息增益比
    """
    calculate gain ration gr(y, x)
    :param x: feature
    :param y: class
    :return: gr
    """
    return gain(x, y)/cal_ent(x)

In [ ]:
df = pd.read_csv("./Input/mdata_5-1.txt", index_col=0)
print(df.head(100))


    年龄 有工作 有自己的房子 信贷情况  类别
ID                        
1   青年   否      否   一般  拒绝
2   青年   否      否    好  拒绝
3   青年   是      否    好  批准
4   青年   是      是   一般  批准
5   青年   否      否   一般  拒绝
6   中年   否      否   一般  拒绝
7   中年   否      否    好  拒绝
8   中年   是      是    好  批准
9   中年   否      是  非常好  批准
10  中年   否      是  非常好  批准
11  老年   否      是  非常好  批准
12  老年   否      是    好  批准
13  老年   是      否    好  批准
14  老年   是      否  非常好  批准
15  老年   否      否   一般  拒绝


In [22]:
# 计算信息量
print(cal_ent(df["类别"]))

def my_cal_ent(x):
    ent = 0.0
    for val in x.values.unique():
        p = x[x==val].shape[0]/x.shape[0];
        ent += -1 * p * np.log2(p)
    return ent

print(my_cal_ent(df["类别"]))

0.9709505944546686
0.9709505944546686


In [42]:
cols = df.columns.tolist()
print(cols)
for col in cols[:-1]:
    print(col, "gain", gain(df[col], df[cols[-1]]))
    print(col, "gain_ratio", gain_ratio(df[col], df[cols[-1]]))

def my_gain(x,y):
    ent = my_cal_ent(y)
    cond_ent = 0.0
    for val in list(set(x)):
        val_row = x==val;
        y_row = y[val_row]
        ment = my_cal_ent(y_row)
        cond_ent += ment * y_row.shape[0]/x.shape[0]
    return ent-cond_ent

def my_gain_ration(x,y):
    gain = my_gain(x,y)
    return gain/my_cal_ent(x)


print(my_gain(df['年龄'],df['类别']))
print(my_gain_ration(df['年龄'],df['类别']))

['年龄', '有工作', '有自己的房子', '信贷情况', '类别']
年龄 gain 0.08300749985576883
年龄 gain_ratio 0.05237190142858302
有工作 gain 0.32365019815155627
有工作 gain_ratio 0.3524465495205019
有自己的房子 gain 0.4199730940219749
有自己的房子 gain_ratio 0.4325380677663126
信贷情况 gain 0.36298956253708536
信贷情况 gain_ratio 0.2318538812872422
0.08300749985576872
0.05237190142858295


In [43]:
X = df[cols[:-1]].values
y = df[cols[-1]].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,shuffle=True,random_state=42)

In [ ]:
clf = Tree(eps=0.02, feas=cols, criterion="gr")
clf.fit(X_train, y_train)
# test ID3 fit predict
print(clf.tree_)


{'有工作__10': {'否__6': {'有工作__6': {'否__4': '拒绝', '是__2': '批准'}}, '是__4': '批准'}}


In [47]:
clf.describe_tree(clf.tree_)
print(clf.predict(X_test))

['拒绝', '拒绝', '拒绝', '批准', '拒绝']


In [49]:
# test cart build EX5.4
cart_tree = clf.create_cart(X, y)
print(cart_tree)

{(2, '否'): {'left': {(1, '否'): {'left': '拒绝', 'right': '批准'}}, 'right': '批准'}}
